This figure is a sequence logo of the binders from the screen

How many counts in the input library should be required for the full library set of lirs and the binder set?
should this PLR sequence be included for LIRs at the N terminus?

In [1]:
from lir_proteome_screen_pssm import environment as env
import pandas as pd
import lir_proteome_screen_pssm.sequence_utils as seqtools
import re
import numpy as np
import copy
import lir_proteome_screen_pssm.data_loaders as dl
from pathlib import Path

In [2]:
def get_regex_matches(s: pd.Series, regex: str):
    matches = list(seqtools.get_regex_matches(regex, s["ID"]))
    # if len(matches) == 0:
    #     return
    return matches

Below is pretty silly. I should've just imported the full table, filtered everything, only kept the single matches, and then split into binders and background. What I did is equivalent to that, but with more steps.

In [ ]:
REGEX = seqtools.regex2overlapping("....[FWY]..[ILV]....")
# REGEX = seqtools.regex2overlapping("...[FWY]..[ILVFWY]")


def import_full_data_table():
    full_data_table = pd.read_csv(env.RAWFILEPATHS.full_screening_table_2, sep=',')
    # full_data_table['ID'] = 'PLR' + full_data_table['ID']
    full_data_table = full_data_table[~full_data_table['ID'].str.contains(r'HPQ', regex=True)]
    full_data_table = full_data_table[full_data_table['ID']!='PLRASQGSDDDWDDEWDDSSTVADEPGALGSGAYPDLDG'] # For this sequence, we know that the actual binding motif is WDDEW from lir_central
    assert full_data_table.duplicated(subset=["ID"], keep=False).sum() == 0, "There should be no duplicate IDs in the full data table"  
    return full_data_table



def import_binders(regex = REGEX, remove_multi_motif_sequences=True):
    '''
    this is super messy, but I want to be able to only change processing in one place
    and not have to change it in multiple cells
    '''
    # ==============================================================================
    # // screening data
    # ==============================================================================
    full_data_table = import_full_data_table()
    # %%
    # ==============================================================================
    # // binders
    # ==============================================================================
    screen_binders_df = full_data_table[full_data_table['avg_z_score'] >= 1.7].copy()
    screen_binders_df = screen_binders_df[screen_binders_df['Input Count'] >= 10].copy()
    screen_binders_df["regex_matches"] = screen_binders_df.apply(get_regex_matches, axis=1, regex=regex)
    screen_binders_df["num_regex_matches"] = screen_binders_df["regex_matches"].apply(lambda x: len(x))
    df_multi = screen_binders_df[screen_binders_df["num_regex_matches"] > 1].copy()
    df_multi = df_multi.explode("regex_matches")
    df_single = screen_binders_df[screen_binders_df["num_regex_matches"] == 1].copy()
    df_single["regex_matches"] = df_single["regex_matches"].apply(lambda x: x[0])
    screen_binders_df = pd.concat([df_multi, df_single])
    screen_binders_df[["lir_sequence", "motif_start", "motif_end"]] = pd.DataFrame(
        screen_binders_df["regex_matches"].tolist(), index=screen_binders_df.index
    )
    if remove_multi_motif_sequences:
        print(f"removing {len(screen_binders_df[screen_binders_df['num_regex_matches'] > 1])} lirs from sequences with multiple motifs")
        check_l = len(screen_binders_df[screen_binders_df["num_regex_matches"] == 1])
        screen_binders_df = screen_binders_df.drop_duplicates(keep=False, subset="ID")
        assert len(screen_binders_df) == check_l, "deduplication yields different length than number with = 1 motif. something is very wrong"
    print(len(screen_binders_df))
    return screen_binders_df


def import_background(regex = REGEX, remove_multi_motif_sequences=True):
    full_data_table = import_full_data_table()
    bg_df = full_data_table[full_data_table['Input Count'] >= 10].copy()
    bg_df["regex_matches"] = bg_df.apply(get_regex_matches, axis=1, regex=regex)
    bg_df["num_regex_matches"] = bg_df["regex_matches"].apply(lambda x: len(x))
    df_multi = bg_df[bg_df["num_regex_matches"] > 1].copy()
    df_multi = df_multi.explode("regex_matches")
    df_single = bg_df[bg_df["num_regex_matches"] == 1].copy()
    df_single["regex_matches"] = df_single["regex_matches"].apply(lambda x: x[0])
    bg_df = pd.concat([df_multi, df_single])
    bg_df[["lir_sequence", "motif_start", "motif_end"]] = pd.DataFrame(
        bg_df["regex_matches"].tolist(), index=bg_df.index
    )
    bg_df["true label"] = 1
    if remove_multi_motif_sequences:
        print(f"removing {len(bg_df[bg_df['num_regex_matches'] > 1])} lirs from sequences with multiple motifs")
        check_l = len(bg_df[bg_df["num_regex_matches"] == 1])
        bg_df = bg_df.drop_duplicates(keep=False, subset="ID")
        assert len(bg_df) == check_l, "deduplication yields different length than number with = 1 motif. something is very wrong"
        assert len(bg_df) == len(df_single), "deduplication yields different length than number with = 1 motif. something is very wrong"
    print(len(bg_df))
    return bg_df


def import_binders_and_background(regex=REGEX, remove_multi_motif_sequences=True):
    screen_binders_df = import_binders(regex=regex, remove_multi_motif_sequences=remove_multi_motif_sequences)
    bg_df = import_background(regex=regex, remove_multi_motif_sequences=remove_multi_motif_sequences)
    return screen_binders_df, bg_df

In [4]:
def round_and_copy_sequence(sequence: str, score: float, scale_factor: int = 10):
    """
    multiply the score by scale_factor and round to the nearest integer
    then return integer copies of the sequence
    """
    rounded_score = max(1, round(score * scale_factor))
    return [sequence] * rounded_score

def scale_seqlist_by_z_score(seqlist: list, z_scores: list, scale_factor: int = 10):
    """
    Scale each sequence in seqlist by its corresponding z-score in z_scores.
    Each sequence is repeated a number of times equal to the rounded z-score multiplied by scale_factor.
    """
    assert len(seqlist) == len(z_scores), "seqlist and z_scores must have the same length"
    scaled_sequences = []
    for seq, z in zip(seqlist, z_scores):
        scaled_seq = round_and_copy_sequence(seq, z, scale_factor)
        scaled_sequences.extend(scaled_seq)
    return scaled_sequences

In [5]:
def get_fg_and_bg_sequences(
    output_folder: str | Path,
    regex: str = REGEX,
    remove_multi_motif_sequences: bool = True,
    scale_factor: int = 10,
):
    screen_binders_df, bg_df = import_binders_and_background(
        regex=regex, remove_multi_motif_sequences=remove_multi_motif_sequences
    )
    output = Path(output_folder)
    output.mkdir(parents=True, exist_ok=True)
    seqscores = screen_binders_df[["lir_sequence", "avg_z_score"]].values
    seqs = list(seqscores[:, 0])
    z_scores = list(seqscores[:, 1])
    fg_scaled = scale_seqlist_by_z_score(seqs, z_scores, scale_factor=scale_factor)
    with open(output / "foreground_sequences.txt", "w") as f:
        for seq in seqs:
            f.write(f"{seq}\n")
    with open(output / "foreground_sequences_scaled.txt", "w") as f:
        for seq in fg_scaled:
            f.write(f"{seq}\n")
    with open(output / "background_sequences.txt", "w") as f:
        for seq in bg_df["lir_sequence"].values:
            f.write(f"{seq}\n")
    # return fg, bg_df["lir_sequence"].tolist()

---

In [6]:
get_fg_and_bg_sequences(
    output_folder="./deduplicated_sequences",
    regex=REGEX,
    remove_multi_motif_sequences=True,
    scale_factor=10,
)

removing 35 lirs from sequences with multiple motifs
136
removing 75046 lirs from sequences with multiple motifs
126019


In [73]:
screen_binders_df, bg_df = import_binders_and_background(regex = REGEX, remove_multi_motif_sequences=True)

removing 35 lirs from sequences with multiple motifs
136
removing 75046 lirs from sequences with multiple motifs
126019


In [45]:
scale_seqlist_by_z_score(['AAAAA', 'BBBBB', 'CCCCC'], [0.1, 0.2, 1.2])

['AAAAA',
 'BBBBB',
 'BBBBB',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC',
 'CCCCC']

In [58]:
screen_binders_df[['lir_sequence','avg_z_score']].values

array([['QDDMWEHIAISM', 3.99],
       ['GDGSWALLTSRT', 3.85],
       ['GEEEFELLLVRL', 3.71],
       ['SDDDWEYLLNSD', 3.67],
       ['SDDDWEYLLNSD', 3.63],
       ['DDDDWEDIMHNL', 3.62],
       ['TQEEWTLLDISQ', 3.55],
       ['TSELFEDLTWFL', 3.53],
       ['TEAEWEDLTQQY', 3.36],
       ['QDEGWETLEVSS', 3.31],
       ['EEEIWEELSVME', 3.3],
       ['ARTLFNQVMEKE', 3.28],
       ['GEEEFELLAGPL', 3.27],
       ['DDEDWGSLEQEA', 3.24],
       ['DDSLYEFLDFVD', 3.24],
       ['KDSGFTIVSPLD', 3.19],
       ['LESGYPFIIISE', 3.18],
       ['SSENWEIIREDE', 3.14],
       ['LFEEFETIPMTW', 3.13],
       ['LWDQFEVLERHT', 3.11],
       ['SSENWEIIREDE', 3.06],
       ['SESDWETLDPSV', 3.03],
       ['QEESWLSVGPGG', 3.02],
       ['GSEEWEDLTSAP', 3.0],
       ['ECYGYDIIIDES', 2.95],
       ['PEDEYELLMPHR', 2.94],
       ['TLNSFTVLETVI', 2.91],
       ['WEEEWTLLGKEE', 2.91],
       ['LLDSYDLLSYDD', 2.85],
       ['HSHAFIDLTEDF', 2.84],
       ['PSASYLEVTPDS', 2.82],
       ['DLDSFSELDSES', 2.8],
       ['TE

In [ ]:
# remove nonbinders from binders and vice versa
print("removing 7mers present in both binders and nonbinders")
print("number of binders in nonbinders")
print(screen_binders_df["7mer"].isin(screen_nonbinders_df["7mer"]).sum())
print("number of nonbinders in binders")
print(screen_nonbinders_df["7mer"].isin(screen_binders_df["7mer"]).sum())
blist = copy.deepcopy(screen_binders_df["7mer"].tolist())
nblist = copy.deepcopy(screen_nonbinders_df["7mer"].tolist())
screen_binders_df = screen_binders_df[~screen_binders_df["7mer"].isin(nblist)]
screen_nonbinders_df = screen_nonbinders_df[~screen_nonbinders_df["7mer"].isin(blist)]